# Exercise 19.1: A simple calcium cycling model

We will implement a simple calcium cycling model based on sympathetic neurons (Friel et al., 1995). The model tracks only two states: the cytosolic calcium $[\mathrm{Ca}^{2+}]_{\mathrm{i}}$ and the SR calcium $[\mathrm{Ca}^{2+}]_{\mathrm{SR}}$.

There are four fluxes:

1. $J_{\mathrm{entry}}$: Calcium entering the cell from outside.
2. $J_{\mathrm{extrusion}}$: Calcium pumped out of the cell.
3. $J_{\mathrm{rel}}$: Calcium released from the SR (CICR).
4. $J_{\mathrm{uptake}}$: Calcium pumped into the SR.

![Friel model schematic](../../fig/friel.svg)

## Exercise 19.1a: Linear fluxes

In the simplest case, we assume the four fluxes are linear and proportional to the concentration gradients. For instance, the entry flux is:

$$
J_{\mathrm{entry}} = k_{\mathrm{entry}} \left( [\mathrm{Ca}^{2+}]_{\mathrm{o}} - [\mathrm{Ca}^{2+}]_{\mathrm{i}} \right)
$$

1. If $[\mathrm{Ca}^{2+}]_{\mathrm{o}} > [\mathrm{Ca}^{2+}]_{\mathrm{i}}$, does $J_{\mathrm{entry}}$ become positive or negative?
2. Write out the release current $J_{\mathrm{rel}}$ so that it is positive when calcium enters the cytosol.
3. The uptake and extrusion currents are _active_ transport. Write them as proportional to the cytosolic concentration alone:
   - $J_{\mathrm{extrusion}} = k_{\mathrm{extrusion}} \times [\mathrm{Ca}^{2+}]_{\mathrm{i}}$
   - $J_{\mathrm{uptake}} = k_{\mathrm{uptake}} \times [\mathrm{Ca}^{2+}]_{\mathrm{i}}$

## Exercise 19.1b: Concentration changes

With the fluxes defined, write out the derivatives of the cytosolic and SR calcium concentrations in terms of the four fluxes. Be careful about the signs.

$$
\frac{\mathrm{d}[\mathrm{Ca}^{2+}]_{\mathrm{i}}}{\mathrm{d}t} = \ldots
$$

$$
\frac{\mathrm{d}[\mathrm{Ca}^{2+}]_{\mathrm{SR}}}{\mathrm{d}t} = \frac{1}{\gamma}\left(\ldots\right)
$$

Here $\gamma = 0.24$ is a volume scaling factor. Because the SR is much smaller than the cytosol, moving a calcium ion into the SR changes its concentration about four times as much.

## Exercise 19.1c: Variable release rate

With purely linear fluxes, it is not possible to find spontaneous calcium oscillations. To produce oscillations, Friel et al. propose making the release rate constant $k_{\mathrm{rel}}$ depend on cytosolic calcium:

$$
k_{\mathrm{rel}} = \kappa_0 + \kappa_1 \frac{[\mathrm{Ca}^{2+}]_{\mathrm{i}}^n}{K_{\mathrm{d}}^n + [\mathrm{Ca}^{2+}]_{\mathrm{i}}^n}
$$

What kind of equation is the second term? Describe qualitatively how $k_{\mathrm{rel}}$ behaves as a function of $[\mathrm{Ca}^{2+}]_{\mathrm{i}}$.

## Exercise 19.1d: Implementing the RHS

Complete the Python code below to define the right-hand side of the ODE system.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


def rhs_friel(t, y, Cao, k_entry, k_extrusion, k_uptake, kappa0, kappa1, Kd, n, gamma):
    # Unpack the state variables
    Cai, CaSR = y

    # Define the linear fluxes
    J_entry = ...
    J_extrusion = ...
    J_uptake = ...

    # Compute the non-linear release rate (CICR)
    k_rel = ...
    J_rel = ...

    # Calculate the derivatives
    dCai_dt = ...
    dCaSR_dt = ...

    return [dCai_dt, dCaSR_dt]

## Exercise 19.1e: Solving and plotting

Use `solve_ivp` to simulate the model for 1000 seconds. Initial conditions: $[\mathrm{Ca}^{2+}]_{\mathrm{i}} = 0.08$ µM, $[\mathrm{Ca}^{2+}]_{\mathrm{SR}} = 4.0$ µM.

Plot $[\mathrm{Ca}^{2+}]_{\mathrm{i}}$ and $[\mathrm{Ca}^{2+}]_{\mathrm{SR}}$ in two separate subplots.

In [ ]:
# Parameters
Cao = 1000.0  # µM
k_entry = 2e-5  # 1/s
k_extrusion = 0.132  # 1/s
k_uptake = 0.9  # 1/s
kappa0 = 0.013  # 1/s
kappa1 = 0.58  # 1/s
Kd = 0.5  # µM
n = 3.0
gamma = 0.24

params = (Cao, k_entry, k_extrusion, k_uptake, kappa0, kappa1, Kd, n, gamma)
y0 = [0.08, 4.0]
t_span = (0, 1000)

# Call the ODE solver
sol = solve_ivp(...)

# Plot the cytosolic calcium over time
plt.subplot(2, 1, 1)
plt.plot(sol.t, sol.y[0])
plt.ylabel("[Ca2+]i (µM)")
plt.xlim(0, 1000)

# Plot the SR calcium over time
plt.subplot(2, 1, 2)
plt.plot(sol.t, ...)
plt.ylabel("[Ca2+]SR (µM)")
plt.xlabel("Time (s)")
plt.xlim(0, 1000)
plt.show()

## Exercise 19.1f: Describing the solution

Explain in broad strokes how the solutions look. Where is calcium moving in the model? Does it seem reasonable that this model can explain a phenomenon called _calcium oscillations_?

## Exercise 19.1g: Plotting the CICR rate

The model is based on _caffeine-induced_ calcium oscillations. Caffeine upregulates the release flux's dependency on cytosolic calcium, making CICR stronger.

Plot the release rate $k_{\mathrm{rel}}$ as a function of $[\mathrm{Ca}^{2+}]_{\mathrm{i}} \in [0, 2]$ µM.

If caffeine makes this function _steeper_, which model parameters could reflect this effect?

In [ ]:
Cai_range = np.linspace(0, 2, 1000)
k_rel = ...

plt.plot(Cai_range, k_rel)
plt.xlabel("[Ca2+]i (µM)")
plt.ylabel("k_rel (1/s)")
plt.xlim(0, 2)
plt.show()

## Exercise 19.1h: Exploring the CICR release rate (Widget)

Below is an interactive widget that plots the release rate $k_{\mathrm{rel}}$ with adjustable parameters: $\kappa_0$, $\kappa_1$, $K_{\mathrm{d}}$, $n$. Play around with the sliders and get a feel for what all four parameters do to the rate function.

If we want a low release rate at low calcium, and a high release rate at high calcium, with a steep and swift transition — which parameters should be low, and which should be high?

In [ ]:
from L10_widget import CICRWidget

CICRWidget().krel_widget()

## Exercise 19.1i: Finding the CICR sweet spot (Widget)

In the paper, the authors state that calcium oscillations are only found at certain levels of caffeine. If there is no caffeine, then the steepness of the CICR function ($k_{\mathrm{rel}}$) is too low, and we get no oscillations. If there is too much caffeine, then the system releases calcium too readily, and we also get no oscillations.

Attempt to shift either $K_{\mathrm{d}}$ (the half-saturation point) or $n$ (the Hill coefficient) to be very high or very low in the widget below.

- Do the oscillations disappear at either extreme?
- Can you find the _sweet spot_ where oscillations appear?

_(The model may take a moment to update. Pull the sliders slowly.)_

In [ ]:
from L10_widget import CICRWidget

CICRWidget().cicr_widget()